# Baseline MBPP Evaluation — CodeGen-350M-mono (350M) — self-guarding version

Evaluates **`Salesforce/codegen-350M-mono`** on MBPP with the *same unified protocol* as the
other baselines: MBPP test split (**n policy = keep-and-score-as-fail**) · greedy · seed 42 ·
new-tokens-only decode · syntax = `ast.parse` (empty = invalid) · full test_list in a
silenced hard-kill sandbox · pass@1 + avg test pass + syntax validity with 95% Wilson CIs ·
JSON with protocol block.

**Why the last two runs failed, and how this version prevents it:** transformers **5.0.0**
(Kaggle's preinstalled version) has a broken CodeGen implementation — first the
`StrictDataclassDefinitionError` on the config, and even past that, generation crashes with
`'CodeGenModel' object has no attribute 'get_head_mask'`. The previous run installed 4.55.2
correctly **but the kernel still had 5.0.0 imported** (the version print showed
`transformers: 5.0.0`), so everything downstream still crashed.

This notebook is self-guarding: **Cell 3 refuses to let anything run on the wrong version.**
If 4.55.2 is on disk but a stale 5.0.0 is imported in the live kernel, it **auto-restarts the
kernel** for you. So the procedure is simply:

1. **Run All.** If the kernel restarts at Cell 3 (it will, on the first attempt in a session
   that ever imported transformers), that is the guard working — not a crash.
2. **Run All again.** Cell 3 now prints `OK` and the whole notebook runs through.


## 1 · Install pinned versions (transformers 5.0.0 cannot run CodeGen)

In [1]:
# transformers 5.0.0 breaks CodeGen twice over (StrictDataclassDefinitionError at config
# load; get_head_mask AttributeError at generate). Pin the last known-good stable.
!pip install -q "transformers==4.55.2" "huggingface_hub>=0.34,<1.0" "tokenizers>=0.21,<0.22"
!pip install -q datasets accelerate tqdm
print("pip done -- Cell 3 will verify the version actually in effect.")

pip done -- Cell 3 will verify the version actually in effect.


## 2 · Version guard — auto-restarts the kernel if a stale transformers is imported

This is the cell that makes a repeat of the last failure impossible. Three outcomes:
- correct version on disk, nothing stale imported → prints **OK**, proceed;
- wrong version on disk → hard stop with instructions (pip cell didn't run/finish);
- correct version on disk but a stale one already imported → **auto-restarts the kernel**;
  when it comes back, just Run All again.


In [2]:
import sys, importlib.metadata as im

REQUIRED = "4.55.2"

disk = im.version("transformers")
print(f"transformers on disk: {disk}")

if disk != REQUIRED:
    raise SystemExit(
        f"transformers {REQUIRED} is NOT what's installed on disk ({disk}). "
        "Run the pip cell above to completion, then run this cell again."
    )

stale = "transformers" in sys.modules and getattr(sys.modules["transformers"], "__version__", "?") != REQUIRED
if stale:
    print(f"A stale transformers {sys.modules['transformers'].__version__} is imported in this kernel.")
    print("AUTO-RESTARTING the kernel now. This is the guard working, not a crash.")
    print(">>> When the kernel comes back: Run All from the top. <<<")
    import os, time
    time.sleep(2)
    os.kill(os.getpid(), 9)   # hard kernel restart

print("OK -- transformers", REQUIRED, "is in effect. Safe to proceed.")

transformers on disk: 4.55.2
OK -- transformers 4.55.2 is in effect. Safe to proceed.


In [3]:
# ======================= CONFIG =======================
MODEL_ID       = "Salesforce/codegen-350M-mono"   # 350M, Python-only (mono), code-specialized
NUM_PROBLEMS   = None           # None = full test split; 20 = smoke test
MAX_NEW_TOKENS = 256
SEED           = 42
TIMEOUT_S      = 5
OUT_FILE       = f"mbpp_baseline_{MODEL_ID.split('/')[-1]}.json"
# ======================================================
print("Evaluating baseline:", MODEL_ID, "->", OUT_FILE)

Evaluating baseline: Salesforce/codegen-350M-mono -> mbpp_baseline_codegen-350M-mono.json


## 3 · Load model + tokenizer

bf16 on GPU with bf16 support, fp16 fallback with a loud warning (the GPT-2-XL
fp16-overflow lesson). `torch_dtype=` is the correct argument name on 4.55.2. The assert
re-checks the imported version one last time before touching the model.


In [4]:
import torch
import transformers
assert transformers.__version__ == "4.55.2", (
    f"Imported transformers is {transformers.__version__}, not 4.55.2 -- "
    "the guard cell above did not run / kernel was not restarted. Run All from the top."
)
print("transformers:", transformers.__version__)

from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token   # CodeGen tokenizer has no pad token

if device == "cuda":
    if torch.cuda.is_bf16_supported():
        dtype = torch.bfloat16
    else:
        dtype = torch.float16
        print("WARNING: no bf16 on this GPU -> fp16 fallback. Check the sanity-check "
              "cell shows a real completion before trusting the full run.")
else:
    dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
ctx_len = getattr(model.config, "max_position_embeddings", None) or getattr(model.config, "n_positions", 2048)
print(f"Loaded {MODEL_ID} — {n_params/1e6:.1f}M params, context {ctx_len}, dtype {dtype}, on {device}")

transformers: 4.55.2


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

Loaded Salesforce/codegen-350M-mono — 356.7M params, context 2048, dtype torch.bfloat16, on cuda


## 4 · MBPP test split + completion-style prompt

In [5]:
from datasets import load_dataset

mbpp = load_dataset("google-research-datasets/mbpp", "full", split="test")
if NUM_PROBLEMS:
    mbpp = mbpp.select(range(min(NUM_PROBLEMS, len(mbpp))))
print(f"MBPP test problems: {len(mbpp)}")

def solution_head(ref_code):
    """Reference solution's leading lines up to and incl. the first `def` line."""
    out = []
    for ln in ref_code.split("\n"):
        out.append(ln)
        if ln.lstrip().startswith("def "):
            return "\n".join(out) + "\n"
    return None

def build_prompt(ex):
    head = solution_head(ex["code"])
    if head is None:
        return None, None
    first_test = ex["test_list"][0] if ex["test_list"] else ""
    prompt = f"# {ex['text']}\n# {first_test}\n{head}"
    return prompt, head

p, h = build_prompt(mbpp[0])
print("--- example prompt ---"); print(p)

MBPP test problems: 500
--- example prompt ---
# Write a python function to remove first and last occurrence of a given character from the string.
# assert remove_Occ("hello","l") == "heo"
def remove_Occ(s,ch): 



## 5 · Sanity check — one raw completion before the full run (mandatory)

Distinguishes the three failure modes before you spend 15–20 min: empty completion (dtype
problem — stop), real-but-unindented (a true model failure like GPT-2-XL — surprising for a
code model, inspect more), indented code (healthy — proceed).


In [6]:
ex = mbpp[0]
prompt, head = build_prompt(ex)
enc = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=64, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
raw = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
print("RAW COMPLETION (repr):")
print(repr(raw))

if not raw.strip():
    print("\n*** EMPTY COMPLETION. STOP -- check the dtype printed at load time. ***")
elif not raw.lstrip("\n").startswith((" ", "\t")):
    print("\n*** WARNING: completion does not start with indentation. ***")
    print("trim_completion() will discard it as an empty body. For a code-trained model")
    print("like CodeGen this would be surprising -- inspect a few more examples before")
    print("trusting a near-0% syntax-validity result.")
else:
    print("\nNon-empty and indented -- looks like a real model output, safe to proceed.")

RAW COMPLETION (repr):
'    if len(s) == 0: \r\n        return "" \r\n    if ch == s[0]: \r\n        return remove_Occ(s[1:],ch) \r\n    else: \r\n        return remove_Occ(s[1:],ch)'

Non-empty and indented -- looks like a real model output, safe to proceed.


## 6 · Sandboxed test execution (child process, silenced, hard kill)

In [7]:
import multiprocessing as mp

def _worker(code_s, setup, tests, q):
    import os as _os, sys as _sys
    _dn = open(_os.devnull, "w"); _sys.stdout = _dn; _sys.stderr = _dn
    ns = {}
    try:
        if setup: exec(setup, ns)
        exec(code_s, ns)
        passed = 0
        for t in tests:
            try: exec(t, ns); passed += 1
            except Exception: pass
        q.put((passed, len(tests)))
    except Exception:
        q.put((0, len(tests)))

def run_tests(code_s, setup, tests, timeout_s=TIMEOUT_S):
    ctx = mp.get_context("fork"); q = ctx.Queue()
    p = ctx.Process(target=_worker, args=(code_s, setup, tests, q))
    p.start(); p.join(timeout_s)
    if p.is_alive():
        p.terminate(); p.join()
        return 0, len(tests)
    try:
        return q.get_nowait()
    except Exception:
        return 0, len(tests)

## 7 · Generate + evaluate

Greedy decoding. Completion cut at the first line that leaves the function. **n policy:
keep-and-score-as-fail** — over-length prompts stay in the set as guaranteed misses,
matching the GPT-2-XL baseline (context here is 2048, so there should be zero of them).


In [8]:
import ast
from tqdm.auto import tqdm

def trim_completion(completion):
    """Keep the function body: stop at the FIRST non-indented, non-blank line
    (including a new top-level `def` -- one function per problem)."""
    kept = []
    for ln in completion.split("\n"):
        if ln.strip() == "":
            kept.append(ln); continue
        if not ln.startswith((" ", "\t")):          # left the function body
            break
        kept.append(ln)
    return "\n".join(kept).rstrip()

results = []
for ex in tqdm(mbpp, desc=f"MBPP baseline [{MODEL_ID.split('/')[-1]}]"):
    prompt, head = build_prompt(ex)
    if prompt is None:
        continue    # no def line in reference -- cannot build a prompt for any model

    enc = tokenizer(prompt, return_tensors="pt").to(device)
    n_in = enc["input_ids"].shape[1]

    if n_in >= ctx_len - MAX_NEW_TOKENS:
        results.append({"task_id": ex["task_id"], "skipped_overlength": True,
                        "syntax_ok": False, "empty": True,
                        "passed": 0, "total": len(ex["test_list"]), "full_pass": False})
        continue

    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    completion = tokenizer.decode(out[0][n_in:], skip_special_tokens=True)  # new tokens only
    body = trim_completion(completion)
    code_str = (head + body).strip()

    if not body.strip():
        s_ok = False   # empty completion counts as invalid
    else:
        try:
            ast.parse(code_str); s_ok = True
        except SyntaxError:
            s_ok = False

    passed, total = run_tests(code_str, ex.get("test_setup_code", "") or "", ex["test_list"])
    results.append({"task_id": ex["task_id"], "skipped_overlength": False,
                    "syntax_ok": s_ok, "empty": not body.strip(),
                    "passed": passed, "total": total,
                    "full_pass": passed == total and total > 0})

print(f"Done: {len(results)} problems evaluated")

n_empty_check = sum(r["empty"] for r in results)
if n_empty_check == len(results):
    print("\n*** WARNING: 100% of completions are empty -- check dtype + sanity cell before trusting. ***")

MBPP baseline [codegen-350M-mono]:   0%|          | 0/500 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Done: 500 problems evaluated


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 8 · Report + save

In [9]:
import json, math

def wilson_ci(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k/n; denom = 1 + z*z/n
    c = (p + z*z/(2*n))/denom
    h = (z/denom)*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (max(0.0, c-h), min(1.0, c+h))

n = len(results)
k_pass    = sum(r["full_pass"] for r in results)
k_syntax  = sum(r["syntax_ok"] for r in results)
n_empty   = sum(r["empty"] for r in results)
n_skipped = sum(r["skipped_overlength"] for r in results)
with_tests = [r for r in results if r["total"] > 0]
pass_at_1 = k_pass/n
syntax_validity = k_syntax/n
avg_test_pass = sum(r["passed"]/r["total"] for r in with_tests)/len(with_tests) if with_tests else 0.0

pass_ci = wilson_ci(k_pass, n)
syntax_ci = wilson_ci(k_syntax, n)

print(f"--- {MODEL_ID} : MBPP baseline (completion-style) ---")
print(f"n = {n} | empty completions = {n_empty} | over-length scored-as-fail = {n_skipped}")
print(f"pass@1:          {pass_at_1*100:.1f}%  (95% CI {pass_ci[0]*100:.1f}% - {pass_ci[1]*100:.1f}%)")
print(f"avg test pass:   {avg_test_pass*100:.1f}%")
print(f"syntax validity: {syntax_validity*100:.1f}%  (95% CI {syntax_ci[0]*100:.1f}% - {syntax_ci[1]*100:.1f}%)")

out = {
    "model_id": MODEL_ID,
    "params_millions": round(n_params/1e6, 1),
    "protocol": {
        "dataset": "MBPP test split (google-research-datasets/mbpp, full)",
        "prompt_style": "completion",
        "n_policy": "keep-and-score-as-fail (over-length scored as failure, not dropped)",
        "decoding": "greedy",
        "seed": SEED,
        "max_new_tokens": MAX_NEW_TOKENS,
        "timeout_s": TIMEOUT_S,
        "dtype": str(dtype),
        "transformers_version": transformers.__version__,
    },
    "n": n, "n_empty": n_empty, "n_skipped_overlength": n_skipped,
    "pass_at_1": pass_at_1, "pass_at_1_ci95": pass_ci,
    "syntax_validity": syntax_validity, "syntax_validity_ci95": syntax_ci,
    "avg_test_pass_rate": avg_test_pass,
    "results": results,
}
with open(OUT_FILE, "w") as f:
    json.dump(out, f, indent=2)
print(f"Saved: {OUT_FILE}")

--- Salesforce/codegen-350M-mono : MBPP baseline (completion-style) ---
n = 500 | empty completions = 0 | over-length scored-as-fail = 0
pass@1:          15.0%  (95% CI 12.1% - 18.4%)
avg test pass:   21.8%
syntax validity: 93.0%  (95% CI 90.4% - 94.9%)
Saved: mbpp_baseline_codegen-350M-mono.json


## Notes

- **If the kernel restarts at the guard cell on your first Run All — that is the fix
  working.** Run All again; the second pass goes straight through.
- Expected result shape: indented, syntactically plausible completions (it's Python-trained),
  syntax validity likely the highest of the small-code set, pass@1 possibly non-trivial —
  published CodeGen-350M-mono numbers are low-double-digit pass@1 on MBPP under few-shot
  protocols, so a lower zero-/one-shot completion-style number here is expected and fine.
- **n policy:** keep-and-score-as-fail (matches GPT-2-XL / OPT-1.3B). CodeParrot-small and
  TinyStarCoder used the old drop policy (n=499) — standardize before the paper table.
- Runtime ≈ 15–20 min on a T4.
- Paper wiring: Table-8 row *CodeGen-350M-mono† 350M* after TinyStarCoder†; figure bar
  between TinyStarCoder and Pretrained.
